# Data analysis - Software Component nr. 2

This notebook will be the support for **Software Component nr. 2**, a document that will contain the following:

1. Analysis of the features used in learning:
    - correlation
    - independence
    - feature importance
2. Data statistics
3. Data distribution
4. Data visualization and interpretation

In [1]:
from pathlib import Path

import pandas as pd

## Converting the raw data to CSV

The UCI HAR dataset ships as whitespace-separated text files, with the features, the labels and the subjects in separate files. The cells below join them into one table per split, `train.csv` and `test.csv`, with the columns `subject`, `activity_id`, `activity` and then the 561 features.

The raw `Inertial Signals` are left out, because the project works on the features that are already extracted.

In [2]:
RAW_DIR = Path.home() / "Documents/Master/Anul I/Sem I/ML/Project/Dataset/human+activity+recognition+using+smartphones/UCI HAR Dataset/UCI HAR Dataset"
CSV_DIR = Path("../data")

CSV_DIR.mkdir(exist_ok=True)

`features.txt` contains 42 names that appear three times each (the `bandsEnergy()` features). Column names must be unique, so these get a suffix with the order of their occurrence: `_1`, `_2`, `_3`. All the other names stay as they are in the dataset.

In [3]:
features = pd.read_csv(RAW_DIR / "features.txt", sep=" ", header=None, names=["index", "name"])["name"]

is_duplicate = features.duplicated(keep=False)
occurrence = features.groupby(features).cumcount() + 1
feature_names = features.where(~is_duplicate, features + "_" + occurrence.astype(str)).tolist()

assert len(set(feature_names)) == 561
print(f"{features[is_duplicate].nunique()} names repeated, {is_duplicate.sum()} columns renamed")
print(features[is_duplicate].value_counts().unique())

activity_labels = pd.read_csv(RAW_DIR / "activity_labels.txt", sep=" ", header=None, index_col=0).iloc[:, 0]
activity_labels

42 names repeated, 126 columns renamed
[3]


0
1               WALKING
2      WALKING_UPSTAIRS
3    WALKING_DOWNSTAIRS
4               SITTING
5              STANDING
6                LAYING
Name: 1, dtype: str

In [4]:
def read_column(path):
    return pd.read_csv(path, header=None).iloc[:, 0]


def load_split(split):
    split_dir = RAW_DIR / split
    X = pd.read_csv(split_dir / f"X_{split}.txt", sep=r"\s+", header=None, names=feature_names)
    activity_id = read_column(split_dir / f"y_{split}.txt")
    meta = pd.DataFrame({
        "subject": read_column(split_dir / f"subject_{split}.txt"),
        "activity_id": activity_id,
        "activity": activity_id.map(activity_labels),
    })
    return pd.concat([meta, X], axis=1)


splits = {split: load_split(split) for split in ["train", "test"]}

for split, df in splits.items():
    df.to_csv(CSV_DIR / f"{split}.csv", index=False)
    print(split, df.shape)

train (7352, 564)


test (2947, 564)


Checking that the CSV files match the data that was loaded from the raw files.

In [5]:
for split, df in splits.items():
    csv = pd.read_csv(CSV_DIR / f"{split}.csv")
    pd.testing.assert_frame_equal(df, csv, check_exact=True)
    assert csv.notna().all().all()
    print(f"{split}: {csv.shape}, {(CSV_DIR / f'{split}.csv').stat().st_size / 1e6:.0f} MB, {csv['subject'].nunique()} subjects")

assert not set(splits["train"]["subject"]) & set(splits["test"]["subject"])

pd.DataFrame({split: df["activity"].value_counts() for split, df in splits.items()})

train: (7352, 564), 48 MB, 21 subjects


test: (2947, 564), 19 MB, 9 subjects


,train,test
activity,,
LAYING,1407,537
SITTING,1286,491
STANDING,1374,532
WALKING,1226,496
WALKING_DOWNSTAIRS,986,420
WALKING_UPSTAIRS,1073,471


In [6]:
splits["train"].iloc[:5, :6]

,subject,activity_id,activity,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z
0,1,5,STANDING,0.288585,-0.020294,-0.132905
1,1,5,STANDING,0.278419,-0.016411,-0.123520
2,1,5,STANDING,0.279653,-0.019467,-0.113462
3,1,5,STANDING,0.279174,-0.026201,-0.123283
4,1,5,STANDING,0.276629,-0.016570,-0.115362
